In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [2]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),

    transforms.ToTensor(), 

    transforms.RandomErasing(p=0.3),

    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [3]:
train_dataset = datasets.ImageFolder("balanced_train", transform=transform)
val_dataset = datasets.ImageFolder("test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

Train: 50505
Val: 7178


In [4]:
model = models.mobilenet_v3_small(pretrained=True)

# stronger classifier
model.classifier[3] = nn.Sequential(
    nn.Linear(model.classifier[3].in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),    
    nn.Linear(256, 7)
)

model = model.to(device)

/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=12)

In [6]:
def train_model(epochs=12):
    best_val = 0

    for epoch in range(epochs):
        print(f"\n===== Epoch {epoch+1}/{epochs} =====")

        model.train()
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        train_acc = 100 * train_correct / train_total

        # validation
        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = 100 * val_correct / val_total

        print(f"Train Acc: {train_acc:.2f}%")
        print(f"Val Acc:   {val_acc:.2f}%")

        scheduler.step()

        # save best model
        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(), "best_emotion_model.pth")
            print(" Best model saved")

train_model(epochs=12)


===== Epoch 1/12 =====
Train Acc: 47.60%
Val Acc:   54.18%
🔥 Best model saved

===== Epoch 2/12 =====
Train Acc: 57.93%
Val Acc:   58.25%
🔥 Best model saved

===== Epoch 3/12 =====
Train Acc: 61.97%
Val Acc:   60.35%
🔥 Best model saved

===== Epoch 4/12 =====
Train Acc: 64.76%
Val Acc:   61.77%
🔥 Best model saved

===== Epoch 5/12 =====
Train Acc: 67.15%
Val Acc:   62.57%
🔥 Best model saved

===== Epoch 6/12 =====
Train Acc: 69.08%
Val Acc:   63.29%
🔥 Best model saved

===== Epoch 7/12 =====
Train Acc: 70.86%
Val Acc:   63.51%
🔥 Best model saved

===== Epoch 8/12 =====
Train Acc: 72.77%
Val Acc:   64.73%
🔥 Best model saved

===== Epoch 9/12 =====
Train Acc: 74.18%
Val Acc:   65.32%
🔥 Best model saved

===== Epoch 10/12 =====
Train Acc: 75.21%
Val Acc:   65.67%
🔥 Best model saved

===== Epoch 11/12 =====
Train Acc: 75.92%
Val Acc:   65.91%
🔥 Best model saved

===== Epoch 12/12 =====
Train Acc: 76.26%
Val Acc:   65.70%
